## Requirements

In [10]:
import os
import glob
import cv2
import numpy as np


import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## 1-Pytorch Dataset Template

In [15]:
class IndustrialDataset(Dataset):
    def __init__(self, image_dir, label_dir, image_size=640):

        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_size = image_size
        self.image_files = sorted(glob.glob(os.path.join(image_dir, "*")))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_path = self.image_files[idx]

        name = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.label_dir, name + ".txt")


        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        image = cv2.resize(image, (self.image_size, self.image_size))
        image = image.astype(np.float32) / 255.0
        image = image.transpose(2, 0, 1)
        image = torch.tensor(image, dtype=torch.float32)


        with open(label_path) as f:
            line = f.readline().strip()

        nums = list(map(float, line.split()))
        cls = int(nums[0])
        bbox = torch.tensor(nums[1:5], dtype=torch.float32)
        kp = nums[5:]
        keypoints = []

        for i in range(0, len(kp), 3):
            x = kp[i]
            y = kp[i + 1]
            v = kp[i + 2]

            keypoints.append([x, y, v])

        keypoints = torch.tensor(keypoints, dtype=torch.float32)
        target = {"class": cls, "bbox": bbox, "keypoints": keypoints}
        return image, target

In [16]:
train_dataset = IndustrialDataset(image_dir="Dataset/train/images", label_dir="Dataset/train/labels", image_size=640)
val_dataset = IndustrialDataset(image_dir="Dataset/valid/images", label_dir="Dataset/valid/labels", image_size=640)
test_dataset = IndustrialDataset(image_dir="Dataset/test/images", label_dir="Dataset/test/labels", image_size=640)

In [18]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

images, targets = next(iter(train_loader))
print(images.shape)
print(targets["bbox"].shape)
print(targets["keypoints"].shape)

torch.Size([16, 3, 640, 640])
torch.Size([16, 4])
torch.Size([16, 2, 3])


## 2-Model